# Embedding models comparison pipeline

In [129]:
import os
from pathlib import Path

from abc import ABC, abstractmethod

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

import transformers
# from tokenizers import BertWordPieceTokenizer
from transformers import BertConfig, BertTokenizer, AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

import sklearn.metrics as skmetrics
from sklearn.metrics import roc_auc_score, average_precision_score

from ray import train, tune
from ray.tune import ResultGrid

from typing import List, Dict, Union, Any, Optional

import re
import random
import itertools
import math
import time
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm.notebook import trange, tqdm
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

In [130]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [131]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


## Modules

### Utils modules

In [132]:
import sklearn.metrics as skmetrics
from sklearn.metrics import confusion_matrix, accuracy_score

def check_input_type(labels: Union[np.ndarray, List[float]], 
                     preds_scores: Union[np.ndarray, List[float]], 
                     threshold: float) -> (np.ndarray, np.ndarray):
    """
    Check and convert input types to numpy arrays and validate their shapes.

    :param labels: Ground truth labels.
    :type labels: Union[np.ndarray, List[float]]
    :param preds_scores: Predicted scores.
    :type preds_scores: Union[np.ndarray, List[float]]
    :param threshold: Threshold for converting scores to binary labels.
    :type threshold: float
    :return: Ground truth labels and predicted scores as numpy arrays.
    :rtype: (np.ndarray, np.ndarray)
    :raises ValueError: If predictions and labels are not in the same shape or not 1D arrays.
    """
    if not isinstance(preds_scores, np.ndarray):
        preds_scores = np.array(preds_scores)
    if not isinstance(labels, np.ndarray):
        labels = np.array(labels)
    if preds_scores.shape != labels.shape:
        raise ValueError("Predictions and labels are not in the same shape")
    if preds_scores.ndim != 1 or labels.ndim != 1:
        raise ValueError("Predictions and labels must be 1D arrays")
    
    return labels, preds_scores

def evaluate_scores(labels: Union[np.ndarray, List[float]], 
                    preds_scores: Union[np.ndarray, List[float]], 
                    eval_metrics: Union[str, List[str]], 
                    threshold: float = 0.5) -> Dict[str, Any]:
    """
    Evaluate prediction scores using specified metrics.

    :param labels: Ground truth labels.
    :type labels: Union[np.ndarray, List[float]]
    :param preds_scores: Predicted scores.
    :type preds_scores: Union[np.ndarray, List[float]]
    :param eval_metrics: Evaluation metrics to be calculated.
    :type eval_metrics: Union[str, List[str]]
    :param threshold: Threshold for converting scores to binary labels, defaults to 0.5.
    :type threshold: float, optional
    :return: Dictionary containing evaluation results.
    :rtype: Dict[str, Any]
    :raises ValueError: If an unsupported metric is provided.
    """
    
    if isinstance(eval_metrics, str): 
        eval_metrics = [eval_metrics]
        
    labels, preds_scores = check_input_type(labels, preds_scores, threshold = threshold)
    preds_labels = apply_threshold(preds_scores, threshold)
    
    results = {}
    unique_classes = np.unique(labels)
    tn, fp, fn, tp = confusion_matrix(labels, preds_labels).ravel()
    for metric in eval_metrics:
        
        if metric == 'confusion_matrix':
            results['tn'], results['fp'], results['fn'], results['tp'] = tn, fp, fn, tp
        elif metric == 'specificity':
            results['specificity'] = tn / (tn+fp)
        elif metric == 'npv':
            results['npv'] = tn / (tn+fn)
        elif metric == 'fnr':
            results['fnr'] = fn / (tp+fn)
        elif metric == 'lift':
            results['lift'] = (tp/(tp+fp))/((tp+fn)/(tp+tn+fp+fn))
        elif 'auc' in metric: # roc_auc, pr_auc
            if len(unique_classes) == 1:
                results[metric] = float('nan')  # or return a default value
                continue
            metric_func = getattr(skmetrics, metric)
            results[metric] = metric_func(labels, preds_scores)
        elif hasattr(skmetrics, metric):
            metric_func = getattr(skmetrics, metric)
            results[metric] = metric_func(labels, preds_labels)
        else:
            raise ValueError(f"Unsupported metric: {metric}")

    return results

def evaluate_scores_epochs(epochs_labels: List[Union[np.ndarray, List[float]]], 
                           epochs_preds_scores: List[Union[np.ndarray, List[float]]], 
                           eval_metrics: Union[str, List[str]],
                           threshold: float = 0.5) -> Dict[str, List[Any]]:
    """
    Evaluate prediction scores for multiple epochs using specified metrics.

    :param epochs_labels: List of ground truth labels for each epoch.
    :type epochs_labels: List[Union[np.ndarray, List[float]]]
    :param epochs_preds_scores: List of predicted scores for each epoch.
    :type epochs_preds_scores: List[Union[np.ndarray, List[float]]]
    :param eval_metrics: Evaluation metrics to be calculated.
    :type eval_metrics: Union[str, List[str]]
    :param threshold: Threshold for converting scores to binary labels, defaults to 0.5.
    :type threshold: float, optional
    :return: Dictionary containing evaluation results for each epoch.
    :rtype: Dict[str, List[Any]]
    :raises ValueError: If the number of epochs in labels and predictions do not match.
    """
    
    if len(epochs_labels) != len(epochs_preds_scores):
        raise ValueError(f"{len(epochs_labelss)} != {len(epochs_preds_scores)}")
        
    metrics_scores = {} 
    for i in range(len(epochs_labels)):
        res = evaluate_scores(epochs_labels[i], 
                             epochs_preds_scores[i], 
                             eval_metrics, 
                             threshold = threshold)
        for k, v in res.items():
            metrics_scores.setdefault(k, []).append(v)
    return metrics_scores

def apply_threshold(values: Union[np.ndarray, List[float]], 
                    threshold: float) -> List[int]:
    """
    Apply a threshold to a list of values to convert them to binary labels.

    :param values: List of values to be thresholded.
    :type values: Union[np.ndarray, List[float]]
    :param threshold: Threshold for converting values to binary labels.
    :type threshold: float
    :return: List of binary labels.
    :rtype: List[int]
    :raises ValueError: If the threshold is not between 0 and 1.
    """
    
    if not 0 < threshold < 1:
        raise ValueError("Percentage must be between 0 and 100")

    # Calculate the number of values to set to 1
    num_positives = int(len(values) * threshold)

    if num_positives == 0:
        return [0] * len(values)
        
    # Find the cutoff value
    sorted_values = sorted(values, reverse=True)
    cutoff_value = sorted_values[num_positives - 1]

    # Create the binary list based on the cutoff value
    binary_list = [1 if value >= cutoff_value else 0 for value in values]

    return binary_list

### Emedding model modules

* `SentenceTransformerEmbedding`
  1. Consider enabling more parameters
  2. Consider allowing finetuning sentenceTransformer object
 
* `GeckoEmbedding`:
  1. Further implement authentication function to allow user to specify access
  2. Reimplement the encode function to allow greater batch size

* `BERTEmbedding`:
  1. Pretrained BERT model

In [29]:
import os
import time
import numpy as np
import torch
from datetime import datetime
from abc import ABC, abstractmethod
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
import vertexai
from vertexai.preview.language_models import TextEmbeddingModel
from typing import List, Union, Optional

class Text2Embedding:

    @abstractmethod
    def encode(self, texts: List[str]) -> np.ndarray:
        """
        Encode a list of texts into embeddings.

        :param texts: List of texts to encode.
        :type texts: List[str]
        :return: Encoded embeddings.
        :rtype: np.ndarray
        """
        pass

    def save_embed(self, 
                   embedding: np.ndarray, 
                   save_path: Optional[str] = None, 
                   file_name: Optional[str] = None) -> None:
        """
        Save the embedding to a file.

        :param embedding: Embedding to save.
        :type embedding: np.ndarray
        :param save_path: Path to save the embedding, defaults to None.
        :type save_path: Optional[str], optional
        :param file_name: Name of the file to save the embedding, defaults to None.
        :type file_name: Optional[str], optional
        """
        
        # check and ensure the specified path exists
        # if it's none then set up default save path
        if save_path is None: 
            save_path = os.path.join(os.getcwd(), 'embedding_outputs')

        # if specified path does not exist, then save to the specified path
        if not os.path.exists(save_path):
            os.makedirs(save_path)

        if file_name[-4:] != '.npy':
            file_name = filename + '.npy '
        
        full_path = os.path.join(save_path, file_name)
        np.save(full_path, embedding)
        print(f"Embedding saved to {full_path}")

class SentenceTransformerEmbedding(Text2Embedding):
    """
    Encode text to embedding using sentence transformer
    """
    def __init__(self, 
                 model_name: str, 
                 multiprocessing: bool = False,
                 **kwargs) -> None:
        """
        Initialize the SentenceTransformerEmbedding class.

        :param model_name: Name of the Sentence Transformer model.
        :type model_name: str
        :param multiprocessing: Whether to use multiprocessing, defaults to False.
        :type multiprocessing: bool, optional
        """
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self.multiprocessing = multiprocessing

    def encode(self, texts: List[str]) -> np.ndarray:
        """
        Encode a list of texts into embeddings.

        :param texts: List of texts to encode.
        :type texts: List[str]
        :return: Encoded embeddings.
        :rtype: np.ndarray
        """
        if self.multiprocessing:
            pool = self.model.start_multi_process_pool()
            embed_output = self.model.encode_multi_process(texts, pool)
            model.stop_multi_process_pool(pool)
        else:
            embed_output = self.model.encode(texts, show_progress_bar = True)
        self.embed_output = embed_output
        return embed_output

    def save_embed(self, save_path: Optional[str] = None, file_name: Optional[str] = None) -> None:
        """
        Save the embedding to a file.

        :param save_path: Path to save the embedding, defaults to None.
        :type save_path: Optional[str], optional
        :param file_name: Name of the file to save the embedding, defaults to None.
        :type file_name: Optional[str], optional
        """
        if not file_name: 
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            file_name = f"ST_embedding_{self.model_name}_{timestamp}.npy"
        super().save_embed(self.embed_output, save_path = save_path, file_name = file_name)

class GeckoEmbedding(Text2Embedding):
    """

    project_name = "anbc-dev-hcm-cm-de"
    location = "us-east4"
    model_name = "textembedding-gecko"
    """
    def __init__(self,
                 model_name: str,
                 project_name: str,
                 location: str) -> None:
        """
        Initialize the GeckoEmbedding class.

        :param model_name: Name of the Gecko model.
        :type model_name: str
        :param project_name: Name of the project.
        :type project_name: str
        :param location: Location of the project.
        :type location: str
        """
        self.model_name = model_name
        self._authentication(project_name=project_name, location=location)
        
    def _authentication(self, project_name: str, location: str) -> None:
        """
        Authenticate with the Vertex AI service.

        :param project_name: Name of the project.
        :type project_name: str
        :param location: Location of the project.
        :type location: str
        """
        try: 
            vertexai.init(project=project_name, 
                          location=location)
        except Exception as e:
            print(e)

    def encode(self, texts: List[str], embed_dimension: Optional[int] = None, batch_size: int = 5) -> np.ndarray:
        """
        Encode a list of texts into embeddings.

        :param texts: List of texts to encode.
        :type texts: List[str]
        :param embed_dimension: Dimension of the embeddings, defaults to None.
        :type embed_dimension: Optional[int], optional
        :param batch_size: Batch size for encoding, defaults to 5.
        :type batch_size: int, optional
        :return: Encoded embeddings.
        :rtype: np.ndarray
        """

        embed_output = []
        start_time = time.time()
        embed_model = TextEmbeddingModel.from_pretrained(self.model_name)
        kwargs = dict(output_dimensionality=embed_dimension) if embed_dimension else {}

        embed_output = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i: i+batch_size]
            embed_output.extend(embed_model.get_embeddings(batch, **kwargs))
            
            if i > 0 and i % 300 == 0 and time.time() - start_time < 60:
                time.sleep(60 - (time.time() - start_time))
                start_time = time.time()
        embed_output = np.asarray([embed_output[i].values for i in range(len(embed_output))])
        self.embed_output = embed_output
        return embed_output

    def save_embed(self, save_path: Optional[str] = None, file_name: Optional[str] = None) -> None:
        """
        Save the embedding to a file.

        :param save_path: Path to save the embedding, defaults to None.
        :type save_path: Optional[str], optional
        :param file_name: Name of the file to save the embedding, defaults to None.
        :type file_name: Optional[str], optional
        """
        if not file_name:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            file_name = f"Gecko_embedding_{self.model_name}_{timestamp}.npy"
        super().save_embed(self.embed_output, save_path = save_path, file_name = file_name)



class BERTEmbedding(Text2Embedding):
    
    def __init__(self,
                 model_name: str,
                 max_length: int,
                 embedding_type: str = 'cls_token') -> None:
        """
        Initialize the BERTEmbedding class.

        :param model_name: Name of the BERT model.
        :type model_name: str
        :param max_length: Maximum length of the tokenized input.
        :type max_length: int
        :param embedding_type: Type of embedding to extract, defaults to 'cls_token'.
        :type embedding_type: str, optional
        """
        self.model_name = model_name
        self.max_length = max_length
        self.embedding_type = embedding_type
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name)

    # here add finetuning and get embedding 

    def encode(self, texts: List[str], batch_size: int = 16) -> np.ndarray:
        """
        Encode a list of texts into embeddings.

        :param texts: List of texts to encode.
        :type texts: List[str]
        :param batch_size: Batch size for encoding, defaults to 16.
        :type batch_size: int, optional
        :return: Encoded embeddings.
        :rtype: np.ndarray
        """
        
        # Convert the list of texts into a DataLoader
        data_loader = DataLoader(texts, batch_size=batch_size)
    
        embed_output = []
        for batch in tqdm(data_loader):
            # Tokenize the batch of texts
            encodings = self.tokenizer.batch_encode_plus(batch, 
                                                        return_tensors='pt', 
                                                        max_length=self.max_length, 
                                                        padding='max_length', 
                                                        truncation=True)
            
            attention_mask = encodings['attention_mask']
            outputs = self.model(input_ids=encodings['input_ids'], 
                                 attention_mask=attention_mask)
            if self.embedding_type == 'pooler_output':
                embeddings = outputs.pooler_output
            elif self.embedding_type == 'cls_token': # cls embedding
                embeddings = outputs.last_hidden_state[:, 0, :]
            elif self.embedding_type == 'last_hidden_state': # avg. token embedding
                # Calculate the average only over non-padding tokens
                mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
                sum_embeddings = torch.sum(outputs.last_hidden_state * mask_expanded, 1)
                sum_mask = mask_expanded.sum(1) # This counts the number of non-padding tokens
                sum_mask = torch.clamp(sum_mask, min=1e-9) # Prevent division by zero
                embeddings = sum_embeddings / sum_mask
            else:
                raise NameError(f"{self.embedding_type} does not exist")
                
            if isinstance(embeddings, torch.Tensor):
                embeddings = np.asarray(embeddings.cpu().detach())
                embed_output.append(embeddings)
        embed_output = np.vstack(embed_output)
        self.embed_output = embed_output
        return embed_output

    def save_embed(self, save_path: Optional[str] = None, file_name: Optional[str] = None) -> None:
        """
        Save the embedding to a file.

        :param save_path: Path to save the embedding, defaults to None.
        :type save_path: Optional[str], optional
        :param file_name: Name of the file to save the embedding, defaults to None.
        :type file_name: Optional[str], optional
        """

        if not file_name:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            file_name = f"BERT_embedding_{self.model_name}_{timestamp}.npy"
        super().save_embed(self.embed_output, save_path = save_path, file_name = file_name)

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
from typing import Sequence, Dict, Any

def encode_texts(texts: Sequence[str],
                 model: Text2Embedding,
                 **kwargs) -> np.ndarray:
    """
    Encode a sequence of texts using a specified model and optionally save the embeddings.

    :param texts: Sequence of texts to encode.
    :type texts: Sequence[str]
    :param model: Model to use for encoding.
    :type model: Text2Embedding
    :param kwargs: Additional keyword arguments for encoding and saving.
    :return: Encoded embeddings.
    :rtype: np.ndarray
    """
    
    embed_outputs = model.encode(texts, **kwargs)
    if 'save_path' in kwargs and 'file_name' in kwargs:
        model.save_embed(save_path = kwargs['save_path'], filename = kwargs['file_name'])
        
    return embed_outputs

#### Test case

In [ ]:
sentences = [
    "Three years later, the coffin was still full of Jello.",
    "The fish dreamed of escaping the fishbowl and into the toilet where he saw his friend go.",
    "The person box was packed with jelly many dozens of months later.",
    "He found a leprechaun in his walnut shell."
]

In [ ]:
# test
model_name = 'bert-base-uncased'
max_length = 512
embedding_type = 'cls_token' # or 'pooler_output'
bert_embedding = BERTEmbedding(model_name, max_length, embedding_type)

embeddings = bert_embedding.encode(sentences)
print(embeddings)

In [ ]:
# test
ste = SentenceTransformerEmbedding(model_name = 'all-MiniLM-L6-v2',
                                  multiprocessing = False)
res = ste.encode(sentences)

### Embedding model registry module

In [ ]:
VALID_MODEL_NAMES = {'hf_bert': ['bert-base-uncased', 'emilyalsentzer/Bio_ClinicalBERT', 'yikuan8/Clinical-Longformer', 'medicalai/ClinicalBERT'],
                    'sentence_transformer': ['all-MiniLM-L6-v2', 'all-mpnet-base-v2', ''],
                    'gecko_embedding': ['text-embedding-004', 'textembedding-gecko@003']}

In [ ]:
class SentenceTransformerEmbeddingModelLoader:
    @staticmethod
    def valid_model_name(model_name):
        if model_name in VALID_MODEL_NAMES['sentence_transformer']:
            return True
        return False

    @staticmethod
    def load_model(model_name, multiprocessing=False, **kwargs):
        return SentenceTransformerEmbedding(model_name, multiprocessing, **kwargs)

class GeckoEmbeddingModelLoader:
    @staticmethod
    def valid_model_name(model_name):
        if model_name in VALID_MODEL_NAMES['gecko_embedding']:
            return True
        return False

    @staticmethod
    def load_model(model_name, project_name, location):
        return GeckoEmbedding(model_name, project_name, location)

class BERTEmbeddingModelLoader:
    @staticmethod
    def valid_model_name(model_name):
        if model_name in VALID_MODEL_NAMES['hf_bert']:
            return True
        return False

    @staticmethod
    def load_model(model_name, 
                   max_length, 
                   embedding_type='last_hidden_state'):
        return BERTEmbedding(model_name, max_length, embedding_type)

class EmbeddingModelRegistry:
    def __init__(self):
        self.loaders = [SentenceTransformerEmbeddingModelLoader, 
                        GeckoEmbeddingModelLoader, 
                        BERTEmbeddingModelLoader]

    def load_model(self, model_name, **kwargs):
        for loader in self.loaders:
            if loader.valid_model_name(model_name):
                try:
                    return loader.load_model(model_name, **kwargs)
                except Exception as e:
                    print(e)
                    continue
        raise NameError(f"The model '{model_name}' does not exist in the known model hubs.")

In [ ]:
embed_model_registry = EmbeddingModelRegistry()
bert_model = embed_model_registry.load_model(model_name = "bert-base-uncased", max_length = 512)

In [ ]:
bert_model

### Dataset module

In [133]:
class EmbeddingDatasets(torch.utils.data.Dataset):

    def __init__(self,
                 embeddings: np.ndarray,
                 labels: np.ndarray) -> None:
        """
        Initialize the EmbeddingDatasets class.

        :param embeddings: The embeddings dataset.
        :type embeddings: np.ndarray
        :param labels: The labels dataset.
        :type labels: np.ndarray
        """
        super().__init__()
        self.embeddings = embeddings
        self.labels = labels
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        """
        Get a sample from the dataset./

        :param idx: The index of the sample.
        :type idx: int
        :return: A dictionary containing the embedding and label.
        :rtype: Dict[str, torch.Tensor]
        """
        return {
            'embedding': torch.tensor(self.embeddings[idx]),
            'label': torch.tensor(self.labels[idx])
        }

### Models

#### DNN model module

In [134]:
class DNNClassifier(nn.Module):
    """
    A generic classification that intake embeddings only (Sentence-transformer, LLM-based model)
    
    """
    def __init__(self, 
                 input_size: Optional[int] = None,
                 hidden_sizes: List[int] = [128, 64],
                 dropout_rate: float = 0.2) -> None:
        """
        Initialize the DNNClassifier class.

        :param input_size: Size of the input embeddings, defaults to None
        :type input_size: Optional[int], optional
        :param hidden_sizes: List of hidden layer sizes, defaults to [128, 64]
        :type hidden_sizes: List[int], optional
        :param dropout_rate: Dropout rate for the dropout layer, defaults to 0.2
        :type dropout_rate: float, optional
        """
        
        super().__init__()
        self.input_size = input_size
        self.hidden_sizes = hidden_sizes
        self.dropout_rate = dropout_rate
        self._build_model()


    def _build_model(self):
        """Builds the model with the current parameters."""
        self._linear_act_layers = nn.ModuleList()
        self._layer_sizes = [self.input_size] + list(self.hidden_sizes) + [1]
        for i in range(len(self._layer_sizes)-1):
            self._linear_act_layers.append(nn.Linear(self._layer_sizes[i], self._layer_sizes[i+1]))
            if i < len(self._layer_sizes)-2:
                self._linear_act_layers.append(nn.ReLU())
                self._linear_act_layers.append(nn.Dropout(self.dropout_rate))        
        self._sigmoid = nn.Sigmoid()
        
    def set_architecture(self,
                         hidden_sizes: List[int],
                         dropout_rate: float) -> None:
        """
        Sets the architecture of the model and rebuilds it.

        :param hidden_sizes: List of hidden layer sizes
        :type hidden_sizes: List[int]
        :param dropout_rate: Dropout rate for the dropout layer
        :type dropout_rate: float
        """
        self.hidden_sizes = hidden_sizes
        self.dropout_rate = dropout_rate
        self._build_model()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs a forward pass through the model.

        :param x: Input tensor
        :type x: torch.Tensor
        :return: Output tensor after passing through the model
        :rtype: torch.Tensor
        """
        for layer in self._linear_act_layers:
            x = layer(x)
        output = self._sigmoid(x)
        return output

#### DNN model test module

In [135]:
import unittest
import torch
from torch import nn
from typing import List

class TestDNNClassifier(unittest.TestCase):
    def setUp(self):
        self.model = DNNClassifier(input_size=10, hidden_sizes=[20, 10], dropout_rate=0.1)

    def test_forward(self):
        input_tensor = torch.randn(1, 10)
        output = self.model(input_tensor)
        self.assertEqual(output.size(), (1, 1))

    def test_set_architecture(self):
        self.model.set_architecture(hidden_sizes=[30, 15], dropout_rate=0.2)
        input_tensor = torch.randn(1, 10)
        output = self.model(input_tensor)
        self.assertEqual(output.size(), (1, 1))

    def test_model_rebuild(self):
        self.model.set_architecture(hidden_sizes=[30, 15], dropout_rate=0.2)
        self.assertEqual(len(self.model._linear_act_layers), 7)  # 3 Linear, 2 Activation, 2 Dropout

unittest.main(argv=['first-arg-is-ignored'], exit=False)

...2024-08-05 15:05:10,968	INFO tune.py:614 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2024-08-05 15:05:16,498	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'hidden_sizes': (512, 256)}
2024-08-05 15:05:21,013	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'hidden_sizes': (256, 128)}
2024-08-05 15:05:21,024	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/jupyter/ray_results/hp_train_func_2024-08-05_15-05-10' in 0.0071s.
2024-08-05 15:05:21,027	INFO tune.py:1039 -- Total run time: 10.06 seconds (10.01 seconds for the tuning loop).


== Status ==
Current time: 2024-08-05 15:05:21 (running for 00:00:10.02)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/32 CPUs, 0/0 GPUs
Current best trial: ebc92b4e with val_loss=0.6918194174766541 and parameters={'hidden_sizes': (512, 256), 'dropout_rate': 0.3886921657415311, 'batch_size': 16, 'lr': 2.1031372406164055e-05, 'optimizer': 'sgd', 'num_epochs': 10, 'momentum': 0.9676794543418447}
Result logdir: /var/tmp/ray/session_2024-07-31_19-46-46_878603_1047025/artifacts/2024-08-05_15-05-10/hp_train_func_2024-08-05_15-05-10/driver_artifacts
Number of trials: 2/2 (2 TERMINATED)




Epochs: 100%|██████████| 2/2 [00:00<00:00, 115.28it/s]
..
----------------------------------------------------------------------
Ran 15 tests in 10.503s

OK


### Trainer

#### Model trainer

In [136]:
import torch
import time
from tqdm import tqdm
import sklearn.metrics as skmetrics
import numpy as np
from typing import List, Tuple, Any

class ModelTrainer:
    """
    A class to train and validate a machine learning model
    """
    def __init__(self, 
                 ml_model: torch.nn.Module, 
                 optimizer: torch.optim.Optimizer, 
                 scheduler: torch.optim.lr_scheduler._LRScheduler, 
                 criterion: torch.nn.Module) -> None:
        """
        Initialize the ModelTrainer class.

        :param ml_model: The machine learning model to be trained.
        :type ml_model: torch.nn.Module
        :param optimizer: The optimizer for training the model.
        :type optimizer: torch.optim.Optimizer
        :param scheduler: The learning rate scheduler.
        :type scheduler: torch.optim.lr_scheduler._LRScheduler
        :param criterion: The loss function.
        :type criterion: torch.nn.Module
        """
        self.ml_model = ml_model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.criterion = criterion
        self.device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')

    def prepare_batch(self, batch: dict) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Prepare a batch of data for training or validation.

        :param batch: A batch of data.
        :type batch: dict
        :return: A tuple containing inputs and labels.
        :rtype: Tuple[torch.Tensor, torch.Tensor]
        """
        inputs = batch['embedding'].to(self.device)
        labels = batch['label'].float().to(self.device)
        
        # Ensure labels are two-dimensional
        if labels.dim() == 1:
            labels = labels.unsqueeze(1)
        return inputs, labels
    
    # def eval_metric_function(self, labels, preds):
    #     unique_classes = np.unique(labels.cpu().numpy())
    #     if len(unique_classes) == 1:
    #         return 0 # or return 0 if you prefer
    #     eval_metric_func = getattr(skmetrics, self.eval_metric)
    #     return eval_metric_func(labels.to(torch.int32).flatten().tolist(), preds.flatten().tolist())
        
    def train_epoch(self, data_loader: torch.utils.data.DataLoader) -> float:
        """
        Train the model for one epoch.

        :param data_loader: DataLoader for the training data.
        :type data_loader: torch.utils.data.DataLoader
        :return: The average training loss for the epoch.
        :rtype: float
        """
        self.ml_model.train()
        total_loss = 0
        total_metric = 0

        for batch in data_loader:
            self.optimizer.zero_grad()
            inputs, labels = self.prepare_batch(batch)
            outputs = self.ml_model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(data_loader)

        return avg_loss

    def validate_epoch(self, data_loader: torch.utils.data.DataLoader) -> Tuple[float, List[int], List[float]]:
        """
        Validate the model for one epoch.

        :param data_loader: DataLoader for the validation data.
        :type data_loader: torch.utils.data.DataLoader
        :return: A tuple containing the average validation loss, labels, and predictions for the epoch.
        :rtype: Tuple[float, List[int], List[float]]
        """
        self.ml_model.eval()
        total_loss = 0
        total_metric = 0
        labels_per_epoch = []
        preds_per_epoch = []
        with torch.no_grad():
            for batch in data_loader:
                inputs, labels = self.prepare_batch(batch)
                outputs = self.ml_model(inputs)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()
                labels_per_epoch.extend(labels.to(torch.int32).flatten().tolist())
                preds_per_epoch.extend(outputs.flatten().tolist())
        
        avg_loss = total_loss / len(data_loader)
        return avg_loss, labels_per_epoch, preds_per_epoch

    def train_epochs(self, 
                     train_dataloader: torch.utils.data.DataLoader, 
                     val_dataloader: torch.utils.data.DataLoader, 
                     num_epochs: int = 10) -> Tuple[List[float], List[float], List[List[int]], List[List[float]]]:
        """
        Train and validate the model for a specified number of epochs.

        :param train_dataloader: DataLoader for the training data.
        :type train_dataloader: torch.utils.data.DataLoader
        :param val_dataloader: DataLoader for the validation data.
        :type val_dataloader: torch.utils.data.DataLoader
        :param num_epochs: Number of epochs to train the model, defaults to 10
        :type num_epochs: int, optional
        :return: A tuple containing average training loss, average validation loss, labels, and predictions for each epoch.
        :rtype: Tuple[List[float], List[float], List[List[int]], List[List[float]]]
        """

        avg_train_loss = []
        avg_val_loss = []
        epochs_preds = []
        epochs_labels = []

        for epoch in tqdm(range(num_epochs), desc="Epochs"):
            train_loss = self.train_epoch(train_dataloader)
            val_loss, val_labels, val_preds = self.validate_epoch(val_dataloader)

            # Update each result
            avg_train_loss.append(train_loss)
            avg_val_loss.append(val_loss)
            epochs_labels.append(val_labels)
            epochs_preds.append(val_preds)
            
        return avg_train_loss, avg_val_loss, epochs_labels, epochs_preds

#### ModelTrainer test cases

In [137]:
import unittest
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from typing import List, Tuple, Any
from tqdm import tqdm

class DummyDataset(Dataset):
    def __init__(self, size: int):
        self.size = size
        self.data = torch.randn(size, 10)
        self.labels = torch.randint(0, 2, (size, 1)).float()

    def __len__(self):
        return self.size

    def __getitem__(self, idx: int):
        return {'embedding': self.data[idx], 'label': self.labels[idx]}

class DummyModel(nn.Module):
    def __init__(self, input_size: int):
        super(DummyModel, self).__init__()
        self.linear = nn.Linear(input_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.linear(x))

class TestModelTrainer(unittest.TestCase):

    def setUp(self):
        self.train_dataset = DummyDataset(size=100)
        self.val_dataset = DummyDataset(size=50)
        self.train_dataloader = DataLoader(self.train_dataset, batch_size=16, shuffle=True)
        self.val_dataloader = DataLoader(self.val_dataset, batch_size=16, shuffle=False)
        self.ml_model = DummyModel(input_size=10)
        self.optimizer = torch.optim.Adam(self.ml_model.parameters(), lr=0.001)
        self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=1, gamma=0.1)
        self.criterion = nn.BCELoss()
        self.trainer = ModelTrainer(ml_model=self.ml_model, optimizer=self.optimizer, scheduler=self.scheduler, criterion=self.criterion)

    def test_initialization(self):
        self.assertEqual(self.trainer.ml_model, self.ml_model)
        self.assertEqual(self.trainer.optimizer, self.optimizer)
        self.assertEqual(self.trainer.scheduler, self.scheduler)
        self.assertEqual(self.trainer.criterion, self.criterion)
        self.assertTrue(isinstance(self.trainer.device, torch.device))

    def test_prepare_batch(self):
        batch = next(iter(self.train_dataloader))
        inputs, labels = self.trainer.prepare_batch(batch)
        self.assertEqual(inputs.shape, (16, 10))
        self.assertEqual(labels.shape, (16, 1))

    def test_train_epoch(self):
        avg_loss = self.trainer.train_epoch(self.train_dataloader)
        self.assertIsInstance(avg_loss, float)

    def test_validate_epoch(self):
        avg_loss, labels, preds = self.trainer.validate_epoch(self.val_dataloader)
        self.assertIsInstance(avg_loss, float)
        self.assertIsInstance(labels, list)
        self.assertIsInstance(preds, list)
        self.assertEqual(len(labels), len(self.val_dataset))
        self.assertEqual(len(preds), len(self.val_dataset))

    def test_train_epochs(self):
        avg_train_loss, avg_val_loss, epochs_labels, epochs_preds = self.trainer.train_epochs(self.train_dataloader, self.val_dataloader, num_epochs=2)
        self.assertIsInstance(avg_train_loss, list)
        self.assertIsInstance(avg_val_loss, list)
        self.assertIsInstance(epochs_labels, list)
        self.assertIsInstance(epochs_preds, list)
        self.assertEqual(len(avg_train_loss), 2)
        self.assertEqual(len(avg_val_loss), 2)
        self.assertEqual(len(epochs_labels), 2)
        self.assertEqual(len(epochs_preds), 2)
unittest.main(argv=['first-arg-is-ignored'], exit=False)

...2024-08-05 15:05:21,517	INFO tune.py:614 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2024-08-05 15:05:27,446	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'hidden_sizes': (256, 128)}
2024-08-05 15:05:31,561	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'hidden_sizes': (512, 256)}
2024-08-05 15:05:31,577	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/jupyter/ray_results/hp_train_func_2024-08-05_15-05-21' in 0.0115s.
2024-08-05 15:05:31,581	INFO tune.py:1039 -- Total run time: 10.06 seconds (10.02 seconds for the tuning loop).


== Status ==
Current time: 2024-08-05 15:05:31 (running for 00:00:10.03)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/32 CPUs, 0/0 GPUs
Current best trial: e7ece946 with val_loss=0.6747452437877655 and parameters={'hidden_sizes': (512, 256), 'dropout_rate': 0.3538642717617214, 'batch_size': 32, 'lr': 0.0005993010310316573, 'optimizer': 'adam', 'num_epochs': 20, 'momentum': 0.9533536405523774}
Result logdir: /var/tmp/ray/session_2024-07-31_19-46-46_878603_1047025/artifacts/2024-08-05_15-05-21/hp_train_func_2024-08-05_15-05-21/driver_artifacts
Number of trials: 2/2 (2 TERMINATED)




Epochs: 100%|██████████| 2/2 [00:00<00:00, 126.79it/s]
..
----------------------------------------------------------------------
Ran 15 tests in 10.796s

OK


### Hyperparameter tuning module

In [138]:
from ray.tune.search.bayesopt import BayesOptSearch
from ray.tune.search.hyperopt import HyperOptSearch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from functools import partial
from ray import tune
from ray.tune import CLIReporter
from ray.tune.schedulers import ASHAScheduler
from typing import List, Dict, Any, Optional

In [139]:
class HyperparameterTuner:
    """ Hyperparameter tune the model
    """
    def __init__(self, 
                 train_dataset: torch.utils.data.Dataset,
                 val_dataset: torch.utils.data.Dataset,
                 ml_model: torch.nn.Module) -> None:
        """
        Initialize the HyperparameterTuner class.

        :param train_dataset: The training dataset.
        :type train_dataset: torch.utils.data.Dataset
        :param val_dataset: The validation dataset.
        :type val_dataset: torch.utils.data.Dataset
        :param ml_model: The machine learning model to be tuned.
        :type ml_model: torch.nn.Module
        """
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.ml_model = ml_model
    
    def hptune(self, 
               param_space: Dict[str, Any],
               eval_metrics: List[str], 
               verbose: int = 1,
               threshold: float = 0.5,
               num_samples: int = 10) -> Any:
        """
        Perform hyperparameter tuning.

        :param param_space: The parameter space for tuning.
        :type param_space: Dict[str, Any]
        :param eval_metrics: List of evaluation metrics.
        :type eval_metrics: List[str]
        :param verbose: Verbosity level, defaults to 1
        :type verbose: int, optional
        :param threshold: Threshold for evaluation, defaults to 0.5
        :type threshold: float, optional
        :param num_samples: Number of samples for tuning, defaults to 10
        :type num_samples: int, optional
        :return: The result grid of the tuning process.
        :rtype: Any
        """

        trainable = partial(
            hp_train_func, 
            train_dataset = self.train_dataset,
            val_dataset = self.val_dataset,
            ml_model = self.ml_model,
            eval_metrics = eval_metrics,
            verbose = verbose
        )
        
        self._configure_tune(num_samples)
        self._configure_run()
        
        tuner = tune.Tuner(
            trainable,
            param_space = param_space,
            run_config = self.run_config,
            tune_config = self.tune_config
        )
        result_grid = tuner.fit()
        return result_grid
        
    def _configure_tune(self, num_samples):
        """
        Configure the hyperparameter tuning process.

        :param num_samples: Number of tuning trials.
        :type num_samples: int
        """
        self.tune_config = tune.TuneConfig(
            search_alg = HyperOptSearch(metric="val_loss", mode="min"),
            mode='min', 
            metric = 'val_loss',  # this metric is the the one reported by train report function
            num_samples=num_samples
        )
    def _configure_run(self) -> None:
        """
        Configure the hyperparameter tuning process.

        :param num_samples: Number of samples for tuning.
        :type num_samples: int
        """
        self.run_config = train.RunConfig(
            progress_reporter=ExperimentTerminationReporter(),
            stop = {"val_loss": 0.2}
        )

class ExperimentTerminationReporter(CLIReporter):
    def should_report(self, trials, done=True):
        """Reports only on experiment termination."""
        return done



def hp_train_func(config: Dict[str, Any], 
                  train_dataset: torch.utils.data.Dataset, 
                  val_dataset: torch.utils.data.Dataset, 
                  ml_model: nn.Module,
                  eval_metrics: List[str],
                  verbose: int,
                  threshold: float = 0.5) -> Dict[str, float]:
    """
    Training function for hyperparameter tuning.

    :param config: Configuration dictionary.
    :type config: Dict[str, Any]
    :param train_dataset: The training dataset.
    :type train_dataset: torch.utils.data.Dataset
    :param val_dataset: The validation dataset.
    :type val_dataset: torch.utils.data.Dataset
    :param ml_model: The machine learning model to be trained.
    :type ml_model: nn.Module
    :param eval_metrics: List of evaluation metrics.
    :type eval_metrics: List[str]
    :param verbose: Verbosity level.
    :type verbose: int
    :param threshold: Threshold for evaluation, defaults to 0.5
    :type threshold: float, optional
    :return: Evaluation results.
    :rtype: Dict[str, float]
    """

    
    train_dataloader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)

    # reset model architecture, hidden size and dropout_rate
    ml_model.set_architecture(hidden_sizes = config['hidden_sizes'],
                           dropout_rate = config['dropout_rate'])

    # tune different types of optimizer and learning rate
    optimizer = _get_optimizer(config, ml_model)
        
    total_steps = len(train_dataloader) * config['num_epochs']
    scheduler = get_linear_schedule_with_warmup(optimizer, 
                                                num_warmup_steps=0, 
                                                num_training_steps=total_steps)
    criterion = nn.BCELoss()
    model_trainer = ModelTrainer(ml_model = ml_model, 
                                 optimizer = optimizer,
                                  scheduler = scheduler,
                                  criterion = criterion)
    
    # check the trainer class and maybe implement another function that only return result_per_epoch; # still use 0.5 to hyperparaemter tuning
    avg_train_loss, avg_val_loss, epochs_labels, epochs_preds_scores = model_trainer.train_epochs(train_dataloader, 
                                                                                             val_dataloader, 
                                                                                             num_epochs = config['num_epochs'])
    # collect each metric for num_epochs {'metric1': [0.1, 0.2...], 'metric2': [0.1, 0.2...]}
    eval_metrics = evaluate_scores_epochs(epochs_labels, 
                                          epochs_preds_scores, 
                                          eval_metrics,
                                          threshold = threshold)
    # report avg. val loss and specified metrics across num_epochs
    hp_eval_results = {'val_loss' : sum(avg_val_loss)/len(avg_val_loss)}
    
    if verbose > 0:
        train.report(hp_eval_results)
    return hp_eval_results

def _get_optimizer(config: Dict[str, Any], ml_model: nn.Module) -> torch.optim.Optimizer:
    """
    Get the optimizer based on the configuration.

    :param config: Configuration dictionary.
    :type config: Dict[str, Any]
    :param ml_model: The machine learning model.
    :type ml_model: nn.Module
    :return: The optimizer.
    :rtype: torch.optim.Optimizer
    """

    if config['optimizer'] == 'adam': 
        return torch.optim.AdamW(ml_model.parameters(), lr=config['lr'])
    elif config['optimizer'] == 'sgd' and 'momentum' in config:
        return torch.optim.SGD(ml_model.parameters(), lr=config['lr'], momentum=config['momentum'])

#### Hyperparameter tuning test module

In [140]:
import unittest
from unittest.mock import MagicMock, patch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from ray import tune
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune import CLIReporter
from functools import partial
from typing import List, Dict, Any

# Assuming the classes and functions are defined in a module named `hyperparameter_tuning`

class DummyDataset(Dataset):
    def __init__(self, size: int):
        self.size = size
        self.data = torch.randn(size, 10)
        self.labels = torch.randint(0, 2, (size, 1)).float()

    def __len__(self):
        return self.size

    def __getitem__(self, idx: int):
        return {'embedding': self.data[idx], 'label': self.labels[idx]}

class TestHyperparameterTuner(unittest.TestCase):

    def setUp(self):
        self.train_dataset = DummyDataset(size=100)
        self.val_dataset = DummyDataset(size=50)
        self.ml_model = DNNClassifier(input_size=10)
        self.tuner = HyperparameterTuner(train_dataset=self.train_dataset, val_dataset=self.val_dataset, ml_model=self.ml_model)

    def test_initialization(self):
        self.assertEqual(self.tuner.train_dataset, self.train_dataset)
        self.assertEqual(self.tuner.val_dataset, self.val_dataset)
        self.assertEqual(self.tuner.ml_model, self.ml_model)

    def test_hptune(self):
        param_space = {
            'hidden_sizes': [64, 32],
            'dropout_rate': 0.3,
            'batch_size': 16,
            'num_epochs': 5,
            'optimizer': 'adam',
            'lr': 0.001
        }
        eval_metrics = ['accuracy']

        # Mocking the Tuner and its fit method
        with unittest.mock.patch('ray.tune.Tuner') as MockTuner:
            mock_tuner_instance = MockTuner.return_value
            mock_tuner_instance.fit.return_value = 'result_grid'

            result = self.tuner.hptune(param_space=param_space, eval_metrics=eval_metrics, num_samples=5)

            self.assertEqual(result, 'result_grid')
            MockTuner.assert_called_once()

    def test_get_optimizer(self):
        config = {
            'optimizer': 'adam',
            'lr': 0.001
        }
        optimizer = _get_optimizer(config, self.ml_model)
        self.assertIsInstance(optimizer, torch.optim.AdamW)

        config = {
            'optimizer': 'sgd',
            'lr': 0.01,
            'momentum': 0.9
        }
        optimizer = _get_optimizer(config, self.ml_model)
        self.assertIsInstance(optimizer, torch.optim.SGD)

class TestExperimentTerminationReporter(unittest.TestCase):

    def test_should_report(self):
        reporter = ExperimentTerminationReporter()
        self.assertTrue(reporter.should_report(trials=[], done=True))
        self.assertFalse(reporter.should_report(trials=[], done=False))

class TestHpTrainFunc(unittest.TestCase):

    def test_hp_train_func(self):
        config = {
            'hidden_sizes': [64, 32],
            'dropout_rate': 0.3,
            'batch_size': 16,
            'num_epochs': 5,
            'optimizer': 'adam',
            'lr': 0.001
        }
        train_dataset = DummyDataset(size=100)
        val_dataset = DummyDataset(size=50)
        ml_model = DNNClassifier(input_size=10)
        eval_metrics = ['lift']
        verbose = 1

        result = hp_train_func(config, train_dataset, val_dataset, ml_model, eval_metrics, verbose)

        self.assertIn('val_loss', result)
        self.assertIsInstance(result['val_loss'], float)
unittest.main(argv=['first-arg-is-ignored'], exit=False)

...2024-08-05 15:05:32,368	INFO tune.py:614 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2024-08-05 15:05:37,825	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'hidden_sizes': (256, 128)}
2024-08-05 15:05:42,194	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'hidden_sizes': (512, 256)}
2024-08-05 15:05:42,206	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/jupyter/ray_results/hp_train_func_2024-08-05_15-05-32' in 0.0072s.
2024-08-05 15:05:42,210	INFO tune.py:1039 -- Total run time: 9.84 seconds (9.80 seconds for the tuning loop).


== Status ==
Current time: 2024-08-05 15:05:42 (running for 00:00:09.81)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/32 CPUs, 0/0 GPUs
Current best trial: 58f6bb78 with val_loss=0.6803709506988526 and parameters={'hidden_sizes': (256, 128), 'dropout_rate': 0.25996143808681305, 'batch_size': 64, 'lr': 0.0001633611690138497, 'optimizer': 'sgd', 'num_epochs': 10, 'momentum': 0.9217086065633754}
Result logdir: /var/tmp/ray/session_2024-07-31_19-46-46_878603_1047025/artifacts/2024-08-05_15-05-32/hp_train_func_2024-08-05_15-05-32/driver_artifacts
Number of trials: 2/2 (2 TERMINATED)




Epochs: 100%|██████████| 2/2 [00:00<00:00, 120.88it/s]
..
----------------------------------------------------------------------
Ran 15 tests in 10.212s

OK


### Evaluation module

In this class, the evaluator is implementated using Classification model module; this will performs the following function
* split (texts, labels)
* encode each set text to embedding
* call classification models and perform (fit, tune, evaluate) function
* return the best ML model for the specified embedding model on the test set, return score

In [141]:
from functools import wraps
from inspect import signature

def filter_kwargs(func):
    """
    Define a decorator that filters **kwargs based on the function's signature.
    This approach automatically removes any keyword arguments that the function does not accept.

    :param func: The function to be decorated.
    :type func: Callable
    :return: The decorated function with filtered keyword arguments.
    :rtype: Callable
    """
    @wraps(func)
    def wrapper(**kwargs):
        sig = signature(func)
        valid_keys = sig.parameters.keys()
        filtered_kwargs = {k: v for k, v in kwargs.items() if k in valid_keys}
        return func(**filtered_kwargs)
    return wrapper


def split_data(X: np.ndarray, 
               y: np.ndarray, 
               test_size: float = 0.1, 
               val_size: float = 0.1, 
               **kwargs: Any) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    
    """
    Splits data into training, testing, and optionally validation sets.

    :param X: Features dataset.
    :type X: np.ndarray
    :param y: Target dataset.
    :type y: np.ndarray
    :param test_size: Proportion of the dataset to include in the test split, defaults to 0.1.
    :type test_size: float, optional
    :param val_size: Proportion of the dataset to include in the validation split, defaults to 0.1.
    :type val_size: float, optional
    :param kwargs: Additional keyword arguments to be passed to train_test_split function.
    :type kwargs: Any
    :return: A tuple containing the split datasets: (X_train, X_val, X_test, y_train, y_val, y_test).
    :rtype: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]
    """
    
    # Adjust test_size for initial split to account for subsequent validation split
    initial_test_size = test_size / (1 - val_size) if val_size + test_size < 1 else test_size
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=initial_test_size, stratify = y, **kwargs)
    # Split the temporary training set into final training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_size / (1 - initial_test_size), stratify = y_temp, **kwargs)
    return X_train, X_val, X_test, y_train, y_val, y_test

In [142]:
class Evaluator(ABC):
    """Base class for all evaluators
    Extend this class and implement perform_evaluation for custom evaluators.
    """

    def __init__(self, 
                 seed: int = 42, 
                 **kwargs):
        self.seed = seed
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        torch.cuda.manual_seed_all(self.seed)
        
    @abstractmethod
    def _prepare_datasets(self):
        pass

    @abstractmethod
    def _tune_hyperparameters(self):
        pass

    @abstractmethod
    def _train_model(self):
        pass

    @abstractmethod
    def _predict(self):
        pass
    
    @abstractmethod
    def perform_evaluation(self):
        pass

class XgboostEvaluator(Evaluator):
    pass

class DNNEvaluator(Evaluator):

    
    def __init__(self,
                 embeddings: np.ndarray,
                 labels: np.ndarray, 
                 rebalance: Union[bool, str] = False,
                 **kwargs: Any) -> None:
        """
        Initialize the DNNEvaluator class.

        :param embeddings: The embeddings dataset.
        :type embeddings: np.ndarray
        :param labels: The labels dataset.
        :type labels: np.ndarray
        :param rebalance: Whether to rebalance the dataset, defaults to False.
                          If a string is provided, it should be the class name of the resampling method from imbalanced-learn.
        :type rebalance: Union[bool, str]
        :param kwargs: Additional keyword arguments.
        """
        super().__init__(**kwargs)
        self.classifier = None
        self.best_config = None
        self._prepare_datasets(embeddings, 
                               labels, 
                               rebalance = rebalance) # Add paramss
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    def _rebalance_dataset(self, 
                           X: np.ndarray, 
                           y: np.ndarray,
                           rebalance: Union[str, bool]) -> Tuple[np.ndarray, np.ndarray]:
        """
        Rebalance the dataset using RandomOverSampler.
 
        :param X: Features dataset.
        :type X: np.ndarray
        :param y: Labels dataset.
        :type y: np.ndarray
        :return: Resampled features and labels.
        :rtype: Tuple[np.ndarray, np.ndarray]
        """
 
        # Dynamically import the resampling class from imbalanced-learn
        try: 
            if rebalance == 'ros':
                sampler = RandomOverSampler(random_state = 44)
            elif rebalance == 'rus':
                sampler = RandomUnderSampler(random_state = 44)
            elif rebalance == 'smoteenn':
                sampler = SMOTEENN(random_state = 44)
        except (ImportError, AttributeError) as e:
            raise ValueError(f"Error importing resampling method {rebalance}: {e} - ros, rus, smoteenn")
        X_resampled, y_resampled = sampler.fit_resample(X, y)
        return X_resampled, y_resampled
    def _prepare_datasets(self, 
                          embeddings: np.ndarray, 
                          labels: np.ndarray, 
                          rebalance: bool,
                          test_size: float = 0.1, 
                         val_size: float = 0.1,
                          **kwargs: Any) -> None:
        """
        Prepare the datasets for training, validation, and testing.
 
        :param embeddings: The embeddings dataset.
        :type embeddings: np.ndarray
        :param labels: The labels dataset.
        :type labels: np.ndarray
        :param rebalance: Whether to rebalance the dataset.
        :type rebalance: bool
        :param kwargs: Additional keyword arguments.
        """
        embedding_train, embedding_val, embedding_test, y_train, y_val, y_test = split_data(embeddings, 
                                                                                            labels, 
                                                                                            **kwargs)
        if rebalance:
            embedding_train, y_train = self._rebalance_dataset(embedding_train, y_train, rebalance)
        self.train_dataset = EmbeddingDatasets(embedding_train.astype('float32'), y_train)
        self.val_dataset = EmbeddingDatasets(embedding_val.astype('float32'), y_val)
        self.test_dataset = EmbeddingDatasets(embedding_test.astype('float32'), y_test)

    def _tune_hyperparameters(self, 
                              param_space: Dict[str, Any],
                              eval_metrics: List[str],
                              num_samples: int,
                              threshold: float,
                              verbose: int) -> Any:
        """
        Tune the hyperparameters of the model.

        :param param_space: The parameter space for hyperparameter tuning.
        :type param_space: dict
        :param eval_metrics: The evaluation metrics.
        :type eval_metrics: list
        :param num_samples: The number of samples for hyperparameter tuning.
        :type num_samples: int
        :param threshold: The threshold for evaluation metrics.
        :type threshold: float
        :param verbose: The verbosity level.
        :type verbose: int
        :return: The results of hyperparameter tuning.
        :rtype: Any
        """
  
        hptuner = HyperparameterTuner(self.train_dataset, 
                                      self.val_dataset, 
                                      self.classifier)
        grid_results = hptuner.hptune(param_space = param_space, 
                                      eval_metrics = eval_metrics,
                                      num_samples = num_samples,
                                      verbose = verbose,
                                      threshold = threshold)
        return grid_results
        
    def _train_model(self, 
                     train_dataloader: DataLoader, 
                     val_dataloader: DataLoader,
                     config: Dict[str, Any]) -> Tuple[List[float], List[float], List[List[int]], List[List[float]]]:
        """
        Train the model.

        :param train_dataloader: DataLoader for the training data.
        :type train_dataloader: DataLoader
        :param val_dataloader: DataLoader for the validation data.
        :type val_dataloader: DataLoader
        :param config: The configuration for training.
        :type config: dict
        :return: Training and validation losses, labels, and predictions for each epoch.
        :rtype: Tuple[List[float], List[float], List[List[int]], List[List[float]]]
        """
        
        total_steps = len(train_dataloader) * config['num_epochs']

        # get the best optimzer based on  the best config 
        optimizer = _get_optimizer(config = config, 
                                   ml_model = self.classifier)
        
        scheduler = get_linear_schedule_with_warmup(optimizer, 
                                                    num_warmup_steps=0, 
                                                    num_training_steps=total_steps)
        criterion = nn.BCELoss()

        model_trainer = ModelTrainer(ml_model = self.classifier, 
                                         optimizer = optimizer,
                                          scheduler = scheduler,
                                          criterion = criterion)
        avg_train_loss, avg_val_loss, epochs_labels, epochs_preds = model_trainer.train_epochs(train_dataloader, 
                                                                                                 val_dataloader,
                                                                                                 num_epochs = config['num_epochs'])
        self.model_trained = True
        return avg_train_loss, avg_val_loss, epochs_labels, epochs_preds

    def _predict(self, 
                 test_dataloader: DataLoader) -> Tuple[List[int], List[float]]:
        """
        Make predictions using the trained model.

        :param test_dataloader: DataLoader for the test data.
        :type test_dataloader: DataLoader
        :return: Test labels and predictions.
        :rtype: Tuple[List[int], List[float]]
        """
        
        self.classifier.eval()
        test_labels = []
        test_preds = []
        with torch.no_grad():
            for batch in test_dataloader:
                inputs = batch['embedding'].to(self.device)
                labels = batch['label'].unsqueeze(1).float().to(self.device)
                outputs = self.classifier(inputs)
                test_labels.extend(labels.to(torch.int32).flatten().tolist())
                test_preds.extend(outputs.flatten().tolist())
        return test_labels, test_preds     

    def get_best_model(self) -> DNNClassifier:
        """
        Get the best model based on hyperparameter tuning.

        :return: The best model.
        :rtype: DNNClassifier
        :raises ValueError: If no best model is available.
        """
        if self.best_config:
            input_size = self.train_dataset[0]['embedding'].shape[0]
            self.classifier = DNNClassifier(input_size = input_size)
            return self.classifier.set_architecture(self.best_config['hidden_sizes'],
                                                     self.best_config['dropout_rate'])
        raise ValueError("No best model is available")
        
   
    
    def perform_evaluation(self, 
                           param_space: Dict[str, Any],
                           eval_metrics: List[str], 
                           threshold: float = 0.35,
                           tuning_verbose: int = 0,
                           num_samples: int = 10) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        """
        Perform the evaluation process.

        :param param_space: The parameter space for hyperparameter tuning.
        :type param_space: dict
        :param eval_metrics: The evaluation metrics.
        :type eval_metrics: list
        :param threshold: The threshold for evaluation metrics, defaults to 0.35.
        :type threshold: float
        :param tuning_verbose: The verbosity level for hyperparameter tuning, defaults to 0.
        :type tuning_verbose: int
        :param num_samples: The number of samples for hyperparameter tuning, defaults to 10.
        :type num_samples: int
        :return: Validation and test metrics.
        :rtype: Tuple[dict, dict]
        """
        
        input_size = self.train_dataset[0]['embedding'].shape[0]
        self.classifier = DNNClassifier(input_size = input_size)
        
        # 1. hyperparameter tuning and get the best configurations
        hp_results = self._tune_hyperparameters(param_space = param_space,
                                                   eval_metrics = eval_metrics, 
                                                   verbose = tuning_verbose,
                                                   threshold = threshold,
                                                   num_samples = num_samples)
        self.best_config = hp_results.get_best_result().config

        # 2. set up the ml model with the best config
        self.classifier.set_architecture(self.best_config['hidden_sizes'],
                                         self.best_config['dropout_rate'])
        
        # 3. create dataloaders that fit the corresponding ml model
        train_dataloader = DataLoader(self.train_dataset, batch_size=self.best_config['batch_size'], shuffle=True)
        val_dataloader = DataLoader(self.val_dataset, batch_size=self.best_config['batch_size'], shuffle=False)
        test_dataloader = DataLoader(self.test_dataset, batch_size=self.best_config['batch_size'], shuffle=False)

        # 4. train and validate with the best model; TODO: free the setting of hp spaces where it's able to identify the parameters and only adjust the specified parameters
        avg_train_loss, avg_val_loss, epochs_labels, epochs_preds_scores = self._train_model(train_dataloader = train_dataloader, 
                                                                                        val_dataloader = val_dataloader, 
                                                                                        config = self.best_config)
        validate_metrics = evaluate_scores_epochs(epochs_labels, 
                                                  epochs_preds_scores, 
                                                  eval_metrics,
                                                  threshold = threshold)
        # 5. predict using test set
        test_labels, test_preds_scores = self._predict(test_dataloader)
        test_metrics = evaluate_scores(test_labels, 
                                       test_preds_scores, 
                                       eval_metrics,
                                      threshold = threshold)
        return validate_metrics, test_metrics

#### Evaluator test module

In [143]:
import unittest
import torch
import numpy as np

class TestDNNEvaluator(unittest.TestCase):
    def setUp(self):
        # Create mock data
        self.embeddings = np.random.rand(100, 10).astype('float32')
        self.labels = np.random.randint(0, 2, 100).astype('float32')

        # Patch the split_data function
        global split_data_func
        split_data_func = split_data

        # Initialize DNNEvaluator
        self.evaluator = DNNEvaluator(embeddings=self.embeddings, labels=self.labels)

    def test_prepare_datasets(self):
        self.assertEqual(len(self.evaluator.train_dataset), 78)
        self.assertEqual(len(self.evaluator.val_dataset), 10)
        self.assertEqual(len(self.evaluator.test_dataset), 12)

    def test_perform_evaluation(self):
        param_space = {
            'hidden_sizes': tune.choice([[128, 64], [256, 128], [512, 256]]),
            'dropout_rate': tune.uniform(0.1, 0.5),
            'batch_size': tune.choice([16, 32, 64]),
            'lr': tune.loguniform(1e-5, 1e-3),
            'optimizer': tune.choice(['adam', 'sgd']),
            'num_epochs': tune.choice([10, 20, 30]),
            'momentum': tune.uniform(0.90, 0.99)
        }
        eval_metrics = [
                         'average_precision_score', 
                         'roc_auc_score', 'precision_score', 
                         'recall_score', 
                         'f1_score', 
                         'accuracy_score']
        val_metrics_per_epoch, test_metrics = self.evaluator.perform_evaluation(eval_metrics=eval_metrics, param_space = param_space, num_samples=2)
        self.assertIsInstance(val_metrics_per_epoch, dict)
        self.assertIsInstance(test_metrics, dict)
        for metric in eval_metrics:
            self.assertIn(metric, test_metrics)
unittest.main(argv=['first-arg-is-ignored'], exit=False)

...2024-08-05 15:05:42,638	INFO tune.py:614 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2024-08-05 15:05:48,385	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'hidden_sizes': (128, 64)}
2024-08-05 15:05:52,227	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'hidden_sizes': (256, 128)}
2024-08-05 15:05:52,238	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/jupyter/ray_results/hp_train_func_2024-08-05_15-05-42' in 0.0073s.
2024-08-05 15:05:52,242	INFO tune.py:1039 -- Total run time: 9.60 seconds (9.56 seconds for the tuning loop).


== Status ==
Current time: 2024-08-05 15:05:52 (running for 00:00:09.57)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/32 CPUs, 0/0 GPUs
Current best trial: 4c817690 with val_loss=0.688895070552826 and parameters={'hidden_sizes': (256, 128), 'dropout_rate': 0.3928110510206705, 'batch_size': 64, 'lr': 0.00010477844255287428, 'optimizer': 'sgd', 'num_epochs': 10, 'momentum': 0.9585971398933125}
Result logdir: /var/tmp/ray/session_2024-07-31_19-46-46_878603_1047025/artifacts/2024-08-05_15-05-42/hp_train_func_2024-08-05_15-05-42/driver_artifacts
Number of trials: 2/2 (2 TERMINATED)




Epochs: 100%|██████████| 2/2 [00:00<00:00, 123.92it/s]
..
----------------------------------------------------------------------
Ran 15 tests in 9.967s

OK


#### Evaluator test case

In [144]:
embeddings = np.random.rand(300, 10).astype('float32')
labels = np.random.randint(0, 2, 300).astype('float32')
param_space = {
    'hidden_sizes': tune.choice([[128, 64], [256, 128], [512, 256]]),
    'dropout_rate': tune.uniform(0.1, 0.5),
    'batch_size': tune.choice([16, 32, 64]),
    'lr': tune.loguniform(1e-3, 1e-1),
    'optimizer': tune.choice(['adam', 'sgd']),
    'num_epochs': tune.choice([10, 20, 30]),
    'momentum': tune.uniform(0.90, 0.99)
}
eval_metrics = ['confusion_matrix', 
                 'average_precision_score', 
                 'roc_auc_score', 
                'precision_score', 
                 'recall_score', 
                 'f1_score', 
                 'accuracy_score']


In [145]:
print(embeddings.shape)
print(labels.shape)

(300, 10)
(300,)


In [146]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [147]:
evaluator = DNNEvaluator(embeddings=embeddings, 
                         labels=labels)
val_metrics, test_metrics = evaluator.perform_evaluation(eval_metrics=eval_metrics, 
                             param_space = param_space,
                             num_samples=1)

2024-08-05 15:05:57,501	INFO tune.py:614 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2024-08-05 15:06:03,250	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'hidden_sizes': (512, 256)}
2024-08-05 15:06:03,261	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/jupyter/ray_results/hp_train_func_2024-08-05_15-05-57' in 0.0064s.
2024-08-05 15:06:03,264	INFO tune.py:1039 -- Total run time: 5.76 seconds (5.71 seconds for the tuning loop).


== Status ==
Current time: 2024-08-05 15:06:03 (running for 00:00:05.72)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/32 CPUs, 0/0 GPUs
Current best trial: 7fdda44a with val_loss=0.6951277315616607 and parameters={'hidden_sizes': (512, 256), 'dropout_rate': 0.4864629737226053, 'batch_size': 64, 'lr': 0.01631513402973216, 'optimizer': 'sgd', 'num_epochs': 10, 'momentum': 0.9844924085209444}
Result logdir: /var/tmp/ray/session_2024-07-31_19-46-46_878603_1047025/artifacts/2024-08-05_15-05-57/hp_train_func_2024-08-05_15-05-57/driver_artifacts
Number of trials: 1/1 (1 TERMINATED)




Epochs: 100%|██████████| 10/10 [00:00<00:00, 70.31it/s]


In [148]:
pd.DataFrame(val_metrics)

,tn,fp,fn,tp,average_precision_score,roc_auc_score,precision_score,recall_score,f1_score,accuracy_score
0,11,4,9,6,0.540000,0.475556,0.6,0.400000,0.48,0.566667
1,11,4,9,6,0.540000,0.488889,0.6,0.400000,0.48,0.566667
2,11,4,9,6,0.540000,0.537778,0.6,0.400000,0.48,0.566667
3,11,4,9,6,0.540000,0.546667,0.6,0.400000,0.48,0.566667
4,12,3,8,7,0.593333,0.600000,0.7,0.466667,0.56,0.633333
5,12,3,8,7,0.593333,0.604444,0.7,0.466667,0.56,0.633333
6,12,3,8,7,0.593333,0.608889,0.7,0.466667,0.56,0.633333
7,12,3,8,7,0.593333,0.608889,0.7,0.466667,0.56,0.633333
8,12,3,8,7,0.593333,0.591111,0.7,0.466667,0.56,0.633333
9,12,3,8,7,0.593333,0.600000,0.7,0.466667,0.56,0.633333


### Data

In [149]:
# # Run "gcloud auth application-default login" on the CMD
# import google.auth
# credentials, project = google.auth.default()

In [150]:
# from google.cloud import bigquery
# cp_3k_orig = "select * from clin_analytics_dec_hcb_dev.a455644_rap_txt_3k_sample_tmp"
# cp_3k_summary_pro = "select * from clin_analytics_dec_hcb_dev.a455644_rap_txt_3k_pro_prompted_summary_tmp"
# cp_3k_summary_flash = "select * from clin_analytics_dec_hcb_dev.a455644_rap_txt_3k_prompted_summary_tmp"
# client = bigquery.Client()
# df_cp_3k_orig = client.query(cp_3k_orig).to_dataframe()
# df_cp_3k_summary_pro = client.query(cp_3k_summary_pro).to_dataframe()
# df_cp_3k_summary_flashy = client.query(cp_3k_summary_flash).to_dataframe()

In [151]:
# # import RAP commercial 3000 samples (Extremely imbalanced)
# df_rap_cp_3k = pd.read_csv("/home/a964286/Thinkubator/rap_cp_3k_notes_summary.csv", index_col = 0)
# labels_90 = df_rap_cp_3k.read30.to_numpy() # 90 days is the focus
# texts = df_rap_cp_3k.clean_txt1.to_list()
# summaries = df_rap_cp_3k.gemini_flash_summary_clean.to_list()

In [152]:
# from sentence_transformers import SentenceTransformer

In [153]:
# # high performance in embedding with relatively acceptable speed
# st_model_name = 'all-MiniLM-L6-v2'#'all-mpnet-base-v2'
# embed_model_registry = EmbeddingModelRegistry()
# st_model = embed_model_registry.load_model(model_name = st_model_name, max_length = 512)

In [154]:
# # Encode the texts
# # Recommend using larger >64G RAM, otherwise it will be very slow
# st_embeddings = st_model.encode(texts[:300])

In [155]:
# # Define hyperparameter space and evaluation metrics
# param_space = {
#     'hidden_sizes': tune.choice([[768], [384], [768, 128], [768, 64], [384, 128], [384, 64]]),
#     'dropout_rate': tune.uniform(0.1, 0.5),
#     'batch_size': tune.choice([16, 32, 64]),
#     'lr': tune.loguniform(1e-3, 1e-1),
#     'optimizer': tune.choice(['adam', 'sgd']),
#     'num_epochs': tune.choice([20, 30, 40]),
#     'momentum': tune.uniform(0.90, 0.99)
# }
# # All available supported metrics
# eval_metrics = ['confusion_matrix', 
#                  'average_precision_score', 
#                  'roc_auc_score', 
#                 'precision_score', 
#                  'recall_score', 
#                  'f1_score', 
#                  'lift',
#                  'accuracy_score']

In [156]:
# st_dnn_evaluator = DNNEvaluator(
#                          embeddings=st_embeddings, 
#                          labels=labels_90[:300], 
#                          param_space=param_space
#                     )
# # Just run hyperparameter tuning once for showcase purpose
# st_val_metrics_per_epoch, st_test_metrics = st_dnn_evaluator.perform_evaluation(eval_metrics=eval_metrics, 
#                                                                                 param_space = param_space,
#                                                                                 num_samples=1)

In [157]:
# st_test_metrics

## Eric's Model Code

#### Pulling data and feature engineering

In [158]:
from google.cloud import bigquery
client = bigquery.Client()

random.seed(35)
np.random.seed(35)

In [159]:
# Get features
sql = """
SELECT
    f.* EXCEPT (asdb_plan_key, post_mnths, first_prv_dt, last_prv_dt, index_dt)
    , e.* EXCEPT (individual_id)
FROM 
    `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_non_embedding_features` AS f
LEFT JOIN
    `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_embeddings` AS e
        ON f.asdb_member_key = e.individual_id
WHERE 1=1
    AND NOT asdb_plan_key IN (33, 54)
    AND post_mnths >= 6
    
"""
df_og = client.query(sql).to_dataframe() 

df_og.shape
#2,542,308 members who qualify with 564 features we are exploring

(2542308, 561)

In [160]:
# Get labels (acute IP) stay (0=no, 1=yes) in the 6 months post-index date
sql = """
SELECT
    asdb_member_key
    , acute_ip_flag
FROM 
    `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_outcome_ip` AS o
WHERE 1=1 
  AND o.asdb_member_key IN (SELECT asdb_member_key FROM `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_non_embedding_features` WHERE 1=1 AND NOT asdb_plan_key IN (33, 54) AND post_mnths >= 6)"""
outcome_og = client.query(sql).to_dataframe() 
outcome_og.shape

(2542308, 2)

In [161]:
from pandas.api.types import is_integer_dtype as is_integer
from pandas.api.types import is_float_dtype as is_float
import re

emb_pattern = r'emb[0-255]+'
emb_col = [col for col in df_og.columns if re.match(emb_pattern ,col)]
df_og[emb_col] = df_og[emb_col].fillna(0)

for c in df_og.columns: 
    dt = df_og[c].dtype
    if is_integer(dt) or is_float(dt):
        df_og[c]=df_og[c].fillna(0) 
        # print("Floatint:", dt)
    else:
        try:
            df_og[c]= df_og[c].fillna('')
        except:
            print("ERROR - DATE VARIABLE FOUND", dt)

In [162]:
df = df_og
outcome = outcome_og

print(df.shape)
print(outcome.shape)

(2542308, 561)
(2542308, 2)


##### Get Categorical Features

###### All features

In [163]:
#  0 := categorical, 1 := continuous, 2 := binary
nem_to_type = {
    'narc':2, 
    # 'otc_fills_yr2': 1,
    # 'otc_fills_yr1':1,  
    'COP':2, 
    'sleep_apnea':2, 
    'spinal_inj':2,
    'back':2,
    'substance':2,
    'ALC':2,
    'bipolar':2,
    'psychoses':2,
    'EDO':2, 
    'SCA':2, 
    'DIA':2, 
    'DEP':2, 
    'abdominal_pain':2, 
    'OST':2, 
    'AID':2, 
    'IDA':2, 
    'ANX':2, 
    'DEM':2, 
    'CYS':2, 
    'autoimmune':2, 
    'MOH':2, 
    'HEM':2, 
    'esrd':2, 
    'HepC':2, 
    'HYP':2, 
    'HYC':2, 
    'immune':2, 
    'intel_dsblty':2, 
    'meta_cancer':2, 
    'liver_dis':2, 
    'MSS':2, 
    'OBE':2, 
    'oud':2, 
    'liver_other':2, 
    'paralysis':2, 
    'PAR':2, 
    'PUD':0, 
    'hmd':2, 
    'PVD':2, 
    'CRO':2, 
    'AST':2, 
    'EPL':2, 
    'low_med_sev_ed_flag_yr2':2, 
    'med_high_sev_ed_flag_yr2':2, 
    'high_sev_ed_flag_yr2':2, 
    'acute_ip_flag_yr1':2, 
    'CHO':2,
    'burns':2, 
    'acute_ip_flag_yr2':2, 
    'cad':2, 
    'Cancer':2, 
    'ed_flag_yr2':2, 
    'high_sev_ed_flag_yr1':2, 
    'med_sev_ed_flag_yr2':2, 
    'AUT':2, 
    'med_high_sev_ed_flag_yr1':2, 
    'low_med_sev_ed_flag_yr1':2, 
    'low_sev_ed_flag_yr1':2, 
    'CBD':2, 
    'CHF':2, 
    'CRF':2, 
    'VNA':2, 
    'CHD':2, 
    'ed_flag_yr1':2, 
    'med_sev_ed_flag_yr1':2, 
    'low_sev_ed_flag_yr2':2, 
    'urbsubr':0, 
    'gender':0, # TODO: IF = M, 1 ELIF = F 0 ELSE NULL 
    'cms_prost_cancer_scrn':0, 
    'cms_hpv_scrn':0, 
    'cms_cvd_scrn':0, 
    'cms_lung_cancer_scrn':0, 
    'cms_pelvic':0,
    'coa_population_group':0, 
    'cms_pap':0, 
    'cms_t2d_scrn':0, 
    'cms_bone_scrn':0, 
    'cms_alc_scrn':0,
    'cms_ibt_cvd':0, 
    'cms_col_scrn':0,
    'index_dt':0, 
    'cms_ibt_obese':0, 
    'cms_flu_vax':0, 
    'cms_pneum_vax':0, 
    'cms_dep_scrn':0,
    'tenure_yr2':1, # either 
    'tenure_yr1':1, # either
    'cms_hepb_vax':0,
    'low_sev_ed_visits_yr2':0, # 7/2/24 removed
    'coa_population_category':0, 
    'cms_mam_scrn':0, # 7/2/24 removed
    'low_sev_ed_visits_yr1':1,
    'sum_acute_ip_admits_yr2':1,
    'sum_acute_ip_admits_yr1':1,
    'cms_tobacco':0,# 7/2/24 removed
    'low_med_sev_ed_visits_yr2':1, 
    'cms_t2d_train':0, # 7/2/24 removed
    'sum_preventable_yr2':1, 
    'cms_hepb_scrn':0, ###
    'sum_preventable_yr1':1, 
    'major_chronic_cnt':1, 
    'low_med_sev_ed_visits_yr1':1,
    'sum_ob':1, 
    'sum_unnecessary_yr2':1, 
    'sum_chol_lab':1, 
    'cms_nutrition':0,  # 7/2/24 removed
    'coe_anesth_clm_yr2':1, 
    'sum_a1c_lab':1, 
    'sum_unnecessary_yr1':1, 
    'sum_avoidable_yr2':1, 
    'gpi2_cnt_yr1':1, 
    'high_sev_ed_visits_yr2':1,
    'gpi2_cnt_yr2':1, 
    'ms_brand_fills_yr2':1, 
    'coe_anesth_clm_yr1':1, 
    'med_sev_ed_visits_yr2':1, 
    'cms_sti_scrn':0, 
    'med_high_sev_ed_visits_yr2':1, 
    'high_sev_ed_visits_yr1':1, 
    'ms_brand_fills_yr1':1, 
    'sum_avoidable_yr1':1, 
    'inhaled_steroid_scripts_yr2':1, 
    'med_high_sev_ed_visits_yr1':1, 
    'med_sev_ed_visits_yr1':1, 
    'inhaled_steroid_scripts_yr1':1, 
    'obs_clm_yr2':1,
    'gpi4_cnt_yr2':1,
    'gpi4_cnt_yr1':1,
    'obs_clm_yr1':1, 
    'sum_dme':1, 
    'antianginal_agent_scripts_yr2':1, 
    'mail_order_fills_yr2':1, 
    'branded_generic_fills_yr2':1, 
    'gpi_cnt_yr1':1,
    'gpi_cnt_yr2':1,
    'adi_score':1, 
    'sdi_score':1, 
    'antianginal_agent_scripts_yr1':1, 
    # 'ethnicity_code', 
    'sum_ed_visits_yr2':1,
    'mail_order_fills_yr1':1,
    'uc_clm_yr2':1, 
    'calcium_channel_blk_scripts_yr2':1,
    'antianxiety_scripts_yr2':1, 
    'beta_blocker_scripts_yr2':1, 
    'sum_ed_visits_yr1':1, 
    'branded_generic_fills_yr1':1,
    'coe_maternity_clm_yr2':1,
    'sum_chemo':1, 
    'agenbr':1, 
    'uc_clm_yr1':1,
    'calcium_channel_blk_scripts_yr1':1,
    'coe_maternity_clm_yr1':1,
    'diuretic_scripts_yr2':1,
    'primarylanguage_desc':0, # TODO: Check counts, limit to other if <2000 ppl
    'beta_blocker_scripts_yr1':1, 
    'antianxiety_scripts_yr1':1,
    'antihypertensive_scripts_yr2':1,
    'ndc_cnt_yr1':1, 
    'ndc_cnt_yr2':1, 
    'lipid_lowering_scripts_yr2':1,
    'diuretic_scripts_yr1':1, 
    'coe_surg_clm_yr2':1, 
    'sum_acute_calc_los_yr1':1, 
    'sum_acute_calc_los_yr2':1,
    'sum_pcp':1, 
    'lipid_lowering_scripts_yr1':1, 
    'antihypertensive_scripts_yr1':1,
    'coe_surg_clm_yr1':1,
    'coe_mrx_clm_yr2':1,
    'antidepressant_scripts_yr2':1,
    'coe_mrx_clm_yr1':1, 
    'antipsychotic_scripts_yr2':1, 
    'anticonvulsant_scripts_yr2':1, 
    'antidiabetic_scripts_yr2':1, 
    'antidepressant_scripts_yr1':1, 
    'coe_radio_clm_yr2':1,
    'anticonvulsant_scripts_yr1':1, 
    'coe_radio_clm_yr1':1, 
    'emis_mrx_clm_yr2':1, 
    'coe_ltc_community_clm_yr1':1, 
    'emis_community_clm_yr1':1,
    'coe_ltc_community_clm_yr2':1, 
    'emis_community_clm_yr2':1, 
    'emis_radio_clm_yr1':1, 
    'emis_radio_clm_yr2':1, 
    'antipsychotic_scripts_yr1':1, 
    'antidiabetic_scripts_yr1':1, 
    'emis_mrx_clm_yr1':1,
    'coe_ip_hos_clm_yr2':1,
    'coe_ip_hos_clm_yr1':1, 
    'coe_ip_non_hos_clm_yr2':1,
    'emis_pcp_clm_yr1':1,
    'emis_pcp_clm_yr2':1,
    'coe_phy_clm_yr1':1, 
    'coe_ip_non_hos_clm_yr1':1,
    'inhaled_steroid_days_supply_yr2':1, 
    'coe_phy_clm_yr2':1, 
    'emis_ip_clm_yr2':1,
    'ss_brand_fills_yr2':1, 
    'emis_ip_clm_yr1':1, 
    'inhaled_steroid_days_supply_yr1':1, 
    'coe_eval_clm_yr2':1, 
    'coe_ltc_ins_clm_yr2':1, 
    'emis_ins_clm_yr2':1, 
    'sum_spec':1, 
    'coe_eval_clm_yr1':1, 
    'ss_brand_fills_yr1':1,
    'retail_fills_yr2':1, 
    'retail_fills_yr1':1, 
    'emis_ins_clm_yr1':1, 
    'coe_ltc_ins_clm_yr1':1, 
    'emis_spec_clm_yr2':1,
    'emis_spec_clm_yr1':1,
    'emis_ed_clm_yr2':1,
    'coe_lab_clm_yr2':1, 
    'generic_fills_yr2':1,
    'emis_ed_clm_yr1':1,
    'last_prv_dt':1, 
    'first_prv_dt':1,
    'coe_lab_clm_yr1':1,
    'maint_drug_fills_yr2':1,
    'emis_lab_clm_yr2':1,
    'formulary_fills_yr2':1,
    'emis_lab_clm_yr1':1, 
    'generic_fills_yr1':1, 
    'coe_op_hos_clm_yr2':1, 
    'formulary_fills_yr1':1,
    'emis_misc_clm_yr2':1,
    'maint_drug_fills_yr1':1,
    'coe_op_hos_clm_yr1':1, 
    'rx_claim_cnt_yr2':1, 
    'antianginal_agent_days_supply_yr2':1,
    'coe_mh_clm_yr2':1,
    'coe_mh_clm_yr1':1,
    'emis_misc_clm_yr1':1, 
    'coe_ltc_home_clm_yr2':1,
    'emis_home_clm_yr2':1,
    'antianginal_agent_days_supply_yr1':1, 
    'ltc_clm_yr2':1,
    'coe_ltc_home_clm_yr1':1,
    'emis_home_clm_yr1':1, 
    'emis_hh_clm_yr2':1, 
    'emis_hh_clm_yr1':1, 
    'ltc_clm_yr1':1, 
    'rx_claim_cnt_yr1':1, 
    'calcium_channel_blk_days_supply_yr2':1, 
    'sum_op_visits_yr2':1, 
    'sum_op_visits_yr1':1, 
    'coe_op_non_hos_clm_yr2':1,
    'emis_ambul_clm_yr2':1, 
    'water_quality':1, 
    'coe_op_non_hos_clm_yr1':1, 
    'calcium_channel_blk_days_supply_yr1':1, 
    'beta_blocker_days_supply_yr2':1,
    'emis_ambul_clm_yr1':1,
    'beta_blocker_days_supply_yr1':1, 
    'emis_mh_clm_yr2':1, 
    'emis_mh_clm_yr1':1, 
    'diuretic_days_supply_yr2':1, 
    'antianxiety_days_supply_yr2':1, 
    'lipid_lowering_days_supply_yr2':1, 
    'coe_other_clm_yr2':1, 
    'coe_other_clm_yr1':1,
    'antianxiety_days_supply_yr1':1, 
    'antihypertensive_days_supply_yr2':1,
    'diuretic_days_supply_yr1':1, 
    'lipid_lowering_days_supply_yr1':1, 
    'antihypertensive_days_supply_yr1':1, 
    'antipsychotic_days_supply_yr2':1,
    'antidepressant_days_supply_yr2':1,
    'antipsychotic_days_supply_yr1':1,
    'antidepressant_days_supply_yr1':1, 
    'anticonvulsant_days_supply_yr2':1, 
    'anticonvulsant_days_supply_yr1':1,
    'antidiabetic_days_supply_yr2':1, 
    'antidiabetic_days_supply_yr1':1, 
    'income_inequality':1, 
    'svi_score':1, 
    'days_supply_sum_yr2':1, 
    'zip_weight_avg_medinc':1,
    'days_supply_sum_yr1':1,
    'food_access':1, 
    'citizenship_index':1,
    'acs_social_risk_score':1, 
    'housing_desert':1, 
    'unemployment_index':1,
    'health_habits':1,
    'natural_disaster':1,
    'proactive_health':1,
    'housing_ownership':1, 
    'health_infra':1,
    'language_score':1, 
    'racial_diversity':1, 
    'housing_quality':1,
    'income_index':1,
    'education_index':1,
    'transport_access':1, 
    'disability_score':1,
    'technology_access':1,
    'poverty_score':1, 
    'social_isolation':1,
    'health_access':1,
    'csdi_social_risk_score':1,
     'asdb_member_key':1,
}

###### Data processing

In [164]:
index_to_feature = dict(enumerate(df.columns))
feature_to_index = {value: key for key, value in index_to_feature.items()}
categorical_features = [feature for feature in nem_to_type if nem_to_type[feature] == 0]
categorical_indices = [feature_to_index[feature] for feature in categorical_features if feature in feature_to_index]
len(categorical_features)

30

In [165]:
# ONE HOT ENCODING

categorical_features.remove('index_dt')
df[categorical_features] = df[categorical_features].astype(str)
df['gender'] = df['gender'].map({'M': 1, 'F': 0}).fillna(-1)  
categorical_features.remove('gender')

In [ ]:
# ONE HOT ENCODING

from sklearn.preprocessing import OneHotEncoder
min_occurrence = 2000

encoder = OneHotEncoder(sparse_output=False)

for feature in categorical_features:
    counts = df[feature].value_counts()
    categories_to_keep = counts[counts >= min_occurrence].index
   
    filtered_df = df[df[feature].isin(categories_to_keep)]
   
    if not filtered_df.empty:
        encoded_data = encoder.fit_transform(filtered_df[[feature]])
        encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out([feature]))
       
        df = df.drop(feature, axis=1)
        df = pd.concat([df, encoded_df], axis=1)

In [ ]:
df.shape

In [ ]:
df = df.set_index('asdb_member_key')
outcome = outcome.set_index('asdb_member_key')
merged = df.merge(outcome, on='asdb_member_key', how='left')

Now we have df and outcome correctly aligned, next step would be to do train/test split. But for DNN we need to convert to np array, convert all to int, random sample for evaluator, and run evaluator

In [ ]:
outcome = outcome.squeeze()

In [ ]:
string_columns = df.select_dtypes(include='object').columns.tolist()

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
for col in string_columns:
    df[col] = label_encoder.fit_transform(df[col])

df= df.astype(float)
outcome= outcome.astype(float)

In [ ]:
# Sample 10,000 then 20,000 for the model (30 min)
sample_indices = random.sample(range(len(df)), 100000)
df_sampled = df.iloc[sample_indices].to_numpy()
outcome_sampled = outcome.iloc[sample_indices].values.squeeze().astype('float32')

#### Hyperparameter Tuning

In [ ]:
param_space = {
    'hidden_sizes': tune.choice([[128, 64], [256, 128], [512, 256]]),
    'dropout_rate': tune.uniform(0.1, 0.5),
    'batch_size': tune.choice([16, 32, 64]),
    'lr': tune.loguniform(1e-3, 1e-1),
    'optimizer': tune.choice(['adam', 'sgd']),
    'num_epochs': tune.choice([10, 20, 30]),
    'momentum': tune.uniform(0.90, 0.99)
}
eval_metrics = ['confusion_matrix', 
                 'average_precision_score', 
                 'roc_auc_score', 
                'precision_score', 
                 'recall_score', 
                 'f1_score', 
                 'accuracy_score']


In [ ]:
print(df_sampled.shape)
print(outcome_sampled.shape)

In [ ]:
df_sampled=df_sampled.astype('float32')
outcome_sampled = outcome_sampled.astype('float32')

In [ ]:
df_sampled = np.where(np.isnan(df_sampled), 0, df_sampled)

In [ ]:
import importlib
from imblearn.under_sampling import RandomUnderSampler
evaluator = DNNEvaluator(embeddings=df_sampled, 
                         labels=outcome_sampled,
                         rebalance = 'rus'
                        )
val_metrics, test_metrics = evaluator.perform_evaluation(eval_metrics=eval_metrics, 
                             param_space = param_space, 
                             num_samples=5)

In [ ]:
best_model = evaluator.classifier
best_model, evaluator.best_config

In [ ]:
test_metrics

In [ ]:
def check_label_distribution(y):
    # Count the number of occurrences of each label 
    count_0 = (y == 0).sum()
    count_1 = (y == 1).sum()
    ratio = count_1 / count_0 if count_0 != 0 else np.inf
    print(count_0, count_1, ratio)
    return count_0, count_1, ratio
check_label_distribution(outcome_sampled)

In [ ]:
PATH = './DNN_models/model_1'
torch.save(best_model.state_dict, PATH)

In [ ]:
input_size = train_dataset[0]['embedding'].shape[0]
args={
    input_size = input_size,
    hidden_sizes = evaluator.best_config.hidden_sizes,
    dropout_rate = evaluator.best_config.dropout_rate
}
model = DNNClassifier(*args, **kwargs) # Arguments???
model.load_state_dict(torch.load(PATH))
model.eval()
# Run evaluations on this model
# Use perform_evaluation again???

In [ ]:
# temp_df = pd.DataFrame(df_sampled)
# temp_df.to_csv("./NN_features_100k.csv")

# temp_outcome = pd.DataFrame(outcome_sampled)
# temp_outcome.to_csv("./NN_labels_100k.csv")

In [ ]:
# Convert input and label matrices with EmbeddingDatasets
# dataset = EmbeddingDatasets(df, outcome)

# Convert to data loader
# dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
import matplotlib.pyplot as plt

# Data
ratios = ['0.03', '0.05', '0.1', '0.2', '0.3', '0.4', '0.5', 'no downsample']
lifts = [19.31, 20.28, 20.03, 20.65, 20.21, 20.00, 19.84, 20.17]

# Create bar chart
plt.figure(figsize=(10, 6))
bars = plt.bar(ratios, lifts, color='tab:blue')

# Adding labels and title
plt.xlabel('Ratio / Method')
plt.ylabel('Average 1% Lift')
plt.title('Average 1% Lift by Downsample Ratio for CatBoost')
plt.ylim(19, 21)

# Adding values on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2 - 0.1, yval + 0.02, round(yval, 2), va='bottom')

# Show plot
plt.show()